In [1]:
import mediapipe as mp

print(mp.__version__)
print(dir(mp))

0.10.9
['CalculatorGraph', 'GraphInputStreamAddMode', 'Image', 'ImageFormat', 'ImageFrame', 'Matrix', 'Packet', 'Timestamp', 'ValidatedGraphConfig', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'calculators', 'packet_creator', 'packet_getter', 'resource_util', 'solutions', 'tasks']


In [2]:
import mediapipe as mp

mp_face_mesh = mp.solutions.face_mesh

In [5]:
import cv2
import mediapipe as mp
import csv

video_path = r"C:\Users\Dell3571\Desktop\딥러닝\샘플영상\good_test.mp4"

mp_face_mesh = mp.solutions.face_mesh

cap = cv2.VideoCapture(video_path)

# 영상 정보
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 출력 영상 설정
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("output_video.mp4", fourcc, fps, (width, height))

# CSV 파일 생성
csv_file = open("face_landmarks.csv", "w", newline="")
csv_writer = csv.writer(csv_file)

# header 생성
header = ["frame"]
for i in range(478):
    header.append(f"x_{i}")
    header.append(f"y_{i}")

csv_writer.writerow(header)

frame_idx = 0

with mp_face_mesh.FaceMesh(
        refine_landmarks=True,
        max_num_faces=1) as face_mesh:

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame_idx += 1

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        row = [frame_idx]

        if result.multi_face_landmarks:

            for face_landmarks in result.multi_face_landmarks:

                for i, lm in enumerate(face_landmarks.landmark):

                    x = int(lm.x * width)
                    y = int(lm.y * height)

                    row.append(x)
                    row.append(y)

                    # 화면에 랜드마크 표시
                    cv2.circle(frame, (x, y), 1, (0,255,0), -1)

        csv_writer.writerow(row)

        out.write(frame)

        cv2.imshow("Face Landmarks", frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
out.release()
csv_file.close()
cv2.destroyAllWindows()